In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalogo_au")

In [0]:
catalogo = dbutils.widgets.get("catalogo")

In [0]:
spark.sql(f"DROP CATALOG IF EXISTS {catalogo} CASCADE;")


In [0]:
spark.sql(f"""CREATE CATALOG IF NOT EXISTS {catalogo}""")

In [0]:
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.raw;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.silver;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.golden;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.exploratory;""")


###Tablas Bronze

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalogo_au.bronze.spotify_tracks (
    track_id       STRING,
    track_name     STRING,
    artist_name    STRING,
    album_name     STRING,
    popularity     DOUBLE,
    duration_ms    LONG,
    explicit       BOOLEAN,
    release_date   STRING,
    danceability   DOUBLE,
    energy         DOUBLE,
    ingestion_date TIMESTAMP
)
USING DELTA;
""")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalogo_au.bronze.spotify_daily_charts (
    track_id       STRING,
    rank           INT,
    country        STRING,
    streams        DOUBLE,
    snapshot_date  DATE,
    ingestion_date TIMESTAMP
)
USING DELTA
PARTITIONED BY (snapshot_date);
""")


###Tablas Silver

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalogo_au.silver.spotify_transformed (
    rank                 INT,
    country              STRING,
    streams              DOUBLE,
    snapshot_date        DATE,
    popularity_category  STRING,
    track_id             STRING,
    track_name           STRING,
    artist_name          STRING,
    album_name           STRING,
    popularity           DOUBLE,
    duration_ms          LONG,
    explicit             BOOLEAN,
    release_date         STRING,
    danceability         DOUBLE,
    energy               DOUBLE,
    days_since_chart     INT,
    bpm_diff             INT,
    market_type          STRING,
    track_duration_type  STRING
)
USING DELTA;
""")



###Tablas Golden

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalogo_au.golden.spotify_final_summary (
    snapshot_date         DATE,
    total_tracks_on_chart LONG,
    max_daily_streams     DOUBLE,
    min_daily_streams     DOUBLE,
    primary_country       STRING,
    market_segment        STRING,
    top_popularity_level  STRING
)
USING DELTA
PARTITIONED BY (snapshot_date);
""")

